In [0]:
# Databricks notebook cell — environment setup (glue, not the star)
%pip install rdflib owlrl
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 3.4 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from rdflib import Graph
from pathlib import Path

# Where the artifact will live in the workspace. Adjust to your repo path.
ARTIFACT_DIR = Path("/Workspace/Users/benoit.cayla@databricks.com/book/CH03")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
TTL_PATH = ARTIFACT_DIR / "customer360.ttl"
print(f"Ontology will be written to: {TTL_PATH}")

Ontology will be written to: /Workspace/Users/benoit.cayla@databricks.com/book/CH03/customer360.ttl


In [0]:
g = Graph()
g.parse(str(TTL_PATH), format="turtle")    # raises on malformed Turtle
print(f"Parsed OK. The ontology contains {len(g)} triples.")

# Sanity check: did all eight classes land?
from rdflib import RDF, OWL
classes = sorted(s.split('#')[-1] for s, _, _ in g.triples((None, RDF.type, OWL.Class)))
print("Declared classes:", classes)


Parsed OK. The ontology contains 128 triples.
Declared classes: ['Account', 'Customer', 'Interaction', 'Order', 'PremiumCustomer', 'Product', 'Region', 'Regulation', 'SupportTicket', 'nfd61c2f84b1e419c8ed057dfb7d92ce5b2']


In [0]:
import owlrl
from rdflib import URIRef, RDF

C360 = "https://ontobricks.example/customer360#"
cust4471 = URIRef(C360 + "Customer4471")
premium  = URIRef(C360 + "PremiumCustomer")
order    = URIRef(C360 + "Order9920")
placedBy = URIRef(C360 + "placedBy")

# Before reasoning: is she premium? (She should NOT be — nobody said so.)
asserted = (cust4471, RDF.type, premium) in g
print(f"Before reasoning, Customer4471 is premium? {asserted}")

# (a) Structural OWL 2 RL closure: subclass, inverse property, etc.
owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(g)
print("Inverse 'placedBy' inferred from 'placed'?",
      (order, placedBy, cust4471) in g)        # True — for free, from owl:inverseOf

# (b) Apply the PremiumCustomer threshold (the equivalentClass axiom),
#     materialized as a SPARQL rule — the regime Chapter 6 formalizes.
g.update("""
    PREFIX c360: <https://ontobricks.example/customer360#>
    INSERT { ?c a c360:PremiumCustomer }
    WHERE  { ?c a c360:Customer ; c360:hasLifetimeValue ?v . FILTER(?v > 15000) }
""")

inferred = (cust4471, RDF.type, premium) in g
print(f"After reasoning,  Customer4471 is premium? {inferred}")

assert not asserted and inferred, \
    "Defining axiom did not fire — check the LTV literal is typed xsd:decimal."
print("VERIFIED: PremiumCustomer was derived, not asserted. The ontology reasons.")

Before reasoning, Customer4471 is premium? False
Inverse 'placedBy' inferred from 'placed'? True
After reasoning,  Customer4471 is premium? True
VERIFIED: PremiumCustomer was derived, not asserted. The ontology reasons.


In [0]:
from rdflib import RDFS, OWL, RDF

# Annotation gate: every class and property must have a label.
missing = []
for s, _, _ in g.triples((None, RDF.type, None)):
    if (s, RDF.type, OWL.Class) in g or \
       (s, RDF.type, OWL.ObjectProperty) in g or \
       (s, RDF.type, OWL.DatatypeProperty) in g:
        if (s, RDFS.label, None) not in g and str(s).startswith(C360):
            missing.append(str(s).split('#')[-1])
assert not missing, f"Unlabeled terms (annotate before shipping): {sorted(set(missing))}"
print("VERIFIED: every Customer 360 class and property is annotated.")

# Publish: write the validated ontology to the workspace path,
# and optionally to a Unity Catalog Volume for downstream chapters.
g_clean = Graph()
g_clean.parse(str(TTL_PATH), format="turtle")        # publish the SOURCE, not the closure
VOLUME_PATH = "/Volumes/benoit_cayla/book/ontology/customer360.ttl"
try:
    dbutils.fs.mkdirs("/Volumes/benoit_cayla/book/ontology")
    g_clean.serialize(destination=VOLUME_PATH, format="turtle")
    print(f"Published to Unity Catalog Volume: {VOLUME_PATH}")
except Exception as e:
    print(f"(Volume publish skipped — workspace copy is authoritative.) {e}")
print(f"Published to workspace: {TTL_PATH}")


VERIFIED: every Customer 360 class and property is annotated.
Published to Unity Catalog Volume: /Volumes/benoit_cayla/book/ontology/customer360.ttl
Published to workspace: /Workspace/Users/benoit.cayla@databricks.com/book/CH03/customer360.ttl
